# Fase 1 - Notebook pareado de NDVI

Notebook pensado para rodar localmente ou no Colab.

Uso local:
- abra o notebook a partir da raiz do repositório;
- confirme que os dados estão em `./data`;
- rode a primeira célula para instalar o projeto em modo editável.

Uso no Colab:
- clone ou envie este repositório para o ambiente;
- ajuste `MONOLITHFARM_PROJECT_DIR` e `MONOLITHFARM_DATA_DIR` se o caminho não for detectado automaticamente;
- rode a primeira célula para instalar o pacote.

Saídas principais desta fase:
- `area_inventory`
- `ndvi_clean`
- `ops_area_daily`
- `miip_daily`
- `pairwise_weekly_features`
- `hypothesis_matrix`


In [ ]:
from __future__ import annotations

import importlib.util
import json
import os
import subprocess
import sys
from pathlib import Path


OUTPUT_SUBDIR = 'notebook_outputs'
NOTEBOOK_MODE = os.environ.get("MONOLITHFARM_NOTEBOOK_MODE", "auto").strip().lower()
if NOTEBOOK_MODE not in {"auto", "jupyter", "colab"}:
    raise ValueError("MONOLITHFARM_NOTEBOOK_MODE deve ser `auto`, `jupyter` ou `colab`.")

PROFILE_NAME = os.environ.get("MONOLITHFARM_PROFILE", "").strip()
CONFIG_ENV_PATH = os.environ.get("MONOLITHFARM_PATHS_FILE", "").strip()
IN_COLAB_RUNTIME = "google.colab" in sys.modules
USE_COLAB_MODE = NOTEBOOK_MODE == "colab" or (NOTEBOOK_MODE == "auto" and IN_COLAB_RUNTIME)
REQUIRED_PACKAGES = ['duckdb', 'pandas', 'plotly', 'pyproj', 'shapely']
COLAB_PROJECT_HINTS = [
    "/content/drive/MyDrive/MonolithFarm",
    "/content/drive/My Drive/MonolithFarm",
    "/content/Projeto-FarmLab",
    "/content/MonolithFarm",
]
COLAB_DATA_HINTS = [f"{project_dir}/data" for project_dir in COLAB_PROJECT_HINTS]


def package_available(name: str) -> bool:
    return importlib.util.find_spec(name) is not None


def first_existing_path(candidates: list[str]) -> Path | None:
    for candidate in candidates:
        path = Path(candidate).expanduser()
        if path.exists():
            return path.resolve()
    return None


def discover_paths_config_file() -> Path | None:
    candidates: list[Path] = []
    if CONFIG_ENV_PATH:
        candidates.append(Path(CONFIG_ENV_PATH).expanduser())

    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        candidates.append(candidate / ".monolithfarm.paths.json")
        candidates.append(candidate / "monolithfarm.paths.json")

    seen: set[str] = set()
    for candidate in candidates:
        resolved = candidate.resolve() if candidate.is_absolute() else candidate
        key = str(resolved)
        if key in seen:
            continue
        seen.add(key)
        if resolved.exists():
            return resolved
    return None


def load_paths_config() -> tuple[dict, Path | None]:
    config_path = discover_paths_config_file()
    if config_path is None:
        return {}, None

    payload = json.loads(config_path.read_text(encoding="utf-8"))
    if not isinstance(payload, dict):
        raise ValueError(f"Arquivo de configuracao invalido: {config_path}")
    return payload, config_path


PATHS_CONFIG, PATHS_CONFIG_FILE = load_paths_config()
CONFIG_BASE_DIR = PATHS_CONFIG_FILE.parent.resolve() if PATHS_CONFIG_FILE is not None else None
DEFAULT_PROFILE = str(PATHS_CONFIG.get("default_profile") or "").strip()
ACTIVE_PROFILE = PROFILE_NAME or DEFAULT_PROFILE
PROFILE_SETTINGS = {}
if ACTIVE_PROFILE:
    profiles = PATHS_CONFIG.get("profiles", {})
    if ACTIVE_PROFILE not in profiles:
        raise KeyError(
            f"Perfil `{ACTIVE_PROFILE}` nao encontrado em {PATHS_CONFIG_FILE}."
        )
    profile_value = profiles.get(ACTIVE_PROFILE)
    if isinstance(profile_value, dict):
        PROFILE_SETTINGS = profile_value
elif isinstance(PATHS_CONFIG.get("profile"), dict):
    PROFILE_SETTINGS = PATHS_CONFIG["profile"]


def config_value(key: str):
    return PROFILE_SETTINGS.get(key) if isinstance(PROFILE_SETTINGS, dict) else None


def resolve_config_path(raw_value: object, *, base_dir: Path | None) -> Path | None:
    if raw_value in {None, ""}:
        return None
    path = Path(str(raw_value)).expanduser()
    if not path.is_absolute() and base_dir is not None:
        path = (base_dir / path).resolve()
    else:
        path = path.resolve()
    return path


def mount_colab_drive_if_needed() -> None:
    if not USE_COLAB_MODE:
        return
    if not IN_COLAB_RUNTIME:
        raise RuntimeError(
            "MONOLITHFARM_NOTEBOOK_MODE='colab' foi definido, mas o runtime atual nao e Google Colab."
        )
    from google.colab import drive

    drive_root = Path("/content/drive")
    drive_root.mkdir(parents=True, exist_ok=True)
    if not (drive_root / "MyDrive").exists() and not (drive_root / "My Drive").exists():
        drive.mount("/content/drive")


def find_project_dir() -> Path:
    explicit = os.environ.get("MONOLITHFARM_PROJECT_DIR")
    if explicit:
        return Path(explicit).expanduser().resolve()

    config_project_dir = resolve_config_path(config_value("project_dir"), base_dir=CONFIG_BASE_DIR)
    if config_project_dir is not None:
        return config_project_dir

    if USE_COLAB_MODE:
        hinted_project = first_existing_path(COLAB_PROJECT_HINTS)
        if hinted_project is not None and (hinted_project / "pyproject.toml").exists():
            return hinted_project
        hinted_data = first_existing_path(COLAB_DATA_HINTS)
        if hinted_data is not None and (hinted_data.parent / "pyproject.toml").exists():
            return hinted_data.parent.resolve()

    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate

    raise FileNotFoundError(
        "Nao foi possivel localizar `pyproject.toml`. Defina MONOLITHFARM_PROJECT_DIR, "
        "MONOLITHFARM_PROFILE ou crie `.monolithfarm.paths.json`."
    )


def find_data_dir(project_dir: Path) -> Path:
    explicit = os.environ.get("MONOLITHFARM_DATA_DIR")
    if explicit:
        return Path(explicit).expanduser().resolve()

    config_data_dir = resolve_config_path(config_value("data_dir"), base_dir=CONFIG_BASE_DIR)
    if config_data_dir is not None:
        return config_data_dir

    if USE_COLAB_MODE:
        hinted_data = first_existing_path(COLAB_DATA_HINTS)
        if hinted_data is not None:
            return hinted_data

    for candidate in [project_dir / "data", project_dir / "FarmLab"]:
        if candidate.exists():
            return candidate.resolve()

    return (project_dir / "data").resolve()


def find_output_dir(project_dir: Path) -> Path:
    explicit = os.environ.get("MONOLITHFARM_OUTPUT_DIR")
    if explicit:
        return Path(explicit).expanduser().resolve()

    config_output_dir = resolve_config_path(config_value("output_dir"), base_dir=CONFIG_BASE_DIR)
    if config_output_dir is not None:
        return config_output_dir

    config_output_root = resolve_config_path(config_value("output_root"), base_dir=CONFIG_BASE_DIR)
    if config_output_root is not None:
        return (config_output_root / OUTPUT_SUBDIR).resolve()

    return (project_dir / OUTPUT_SUBDIR).resolve()


def auto_install_enabled() -> bool:
    explicit = os.environ.get("MONOLITHFARM_AUTO_INSTALL")
    if explicit is not None:
        return explicit == "1"

    config_auto_install = config_value("auto_install")
    if config_auto_install is not None:
        return bool(config_auto_install)

    return USE_COLAB_MODE


mount_colab_drive_if_needed()
PROJECT_DIR = find_project_dir()
DATA_DIR = find_data_dir(PROJECT_DIR)
OUTPUT_DIR = find_output_dir(PROJECT_DIR)
AUTO_INSTALL = auto_install_enabled()

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

if AUTO_INSTALL and any(not package_available(name) for name in REQUIRED_PACKAGES):
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(PROJECT_DIR)])
    except Exception:
        subprocess.check_call(["uv", "pip", "install", "--python", sys.executable, "-e", str(PROJECT_DIR)])

print("NOTEBOOK_MODE =", NOTEBOOK_MODE)
print("IN_COLAB_RUNTIME =", IN_COLAB_RUNTIME)
print("USE_COLAB_MODE =", USE_COLAB_MODE)
print("PROFILE_NAME =", PROFILE_NAME or "<default>")
print("PATHS_CONFIG_FILE =", PATHS_CONFIG_FILE)
print("ACTIVE_PROFILE =", ACTIVE_PROFILE or "<none>")
print("PROJECT_DIR =", PROJECT_DIR)
print("DATA_DIR    =", DATA_DIR)
print("OUTPUT_DIR  =", OUTPUT_DIR)
print("AUTO_INSTALL =", AUTO_INSTALL)

if not DATA_DIR.exists():
    raise FileNotFoundError(f"Diretorio de dados nao encontrado: {DATA_DIR}")


In [ ]:
import pandas as pd
import plotly.express as px
from IPython.display import Image as NotebookImage
from IPython.display import Markdown, display

from farmlab.pairwise import OUTPUT_TABLES, build_phase1_workspace, save_phase1_outputs


workspace = build_phase1_workspace(DATA_DIR)
area_inventory = workspace["area_inventory"]
ndvi_clean = workspace["ndvi_clean"]
ops_area_daily = workspace["ops_area_daily"]
miip_daily = workspace["miip_daily"]
pairwise_weekly_features = workspace["pairwise_weekly_features"]
hypothesis_matrix = workspace["hypothesis_matrix"]

print("Tabelas prontas:", ", ".join(OUTPUT_TABLES))


In [ ]:
def sample_rows_per_area(frame: pd.DataFrame, max_images_per_area: int = 3) -> pd.DataFrame:
    if frame.empty:
        return frame.copy()

    sampled_groups = []
    for _, group in frame.sort_values("date").groupby("season_id", sort=False):
        if len(group) <= max_images_per_area:
            sampled_groups.append(group)
            continue

        index_positions = sorted({0, len(group) // 2, len(group) - 1})
        sampled_groups.append(group.iloc[index_positions[:max_images_per_area]])

    return pd.concat(sampled_groups, ignore_index=True)


def closest_rows_by_date(frame: pd.DataFrame, target_date: str) -> pd.DataFrame:
    if frame.empty:
        return frame.copy()

    target = pd.Timestamp(target_date)
    rows = []
    for _, group in frame.sort_values("date").groupby("season_id", sort=False):
        candidate = group.loc[(group["date"] - target).abs().idxmin()]
        rows.append(candidate)
    return pd.DataFrame(rows).sort_values(["comparison_pair", "area_label"]).reset_index(drop=True)


def display_ndvi_gallery(frame: pd.DataFrame, title: str, *, width: int = 420) -> None:
    display(Markdown(f"## {title}"))
    if frame.empty:
        print("Sem imagens NDVI disponiveis para a selecao.")
        return

    for area_label, group in frame.sort_values(["comparison_pair", "area_label", "date"]).groupby("area_label", sort=False):
        area_info = group.iloc[0]
        display(Markdown(f"### {area_label}"))
        display(
            group[
                [
                    "date",
                    "comparison_pair",
                    "treatment",
                    "ndvi_mean",
                    "soil_pct",
                    "dense_veg_pct",
                    "b1_valid_pixels",
                ]
            ].reset_index(drop=True)
        )
        for row in group.itertuples(index=False):
            caption = (
                f"{pd.Timestamp(row.date).date()} | "
                f"NDVI medio={row.ndvi_mean:.3f} | "
                f"solo={row.soil_pct:.1f}% | "
                f"veg densa={row.dense_veg_pct:.1f}%"
            )
            display(Markdown(caption))
            if pd.notna(row.image_path):
                display(NotebookImage(filename=str(row.image_path), width=width))
            else:
                print("Imagem JPG nao encontrada para esta cena.")
    display(Markdown("> As imagens JPG servem para inspecao visual. A analise quantitativa continua usando o CSV de metadados."))


In [ ]:
display(Markdown("## Inventario das areas"))
display(area_inventory)

display(Markdown("## Lacunas atuais"))
display(pd.DataFrame({"gap": workspace["gaps"]}))


In [ ]:
display(Markdown("## Evolucao semanal de NDVI por par"))

if pairwise_weekly_features.empty:
    print("Nao ha features semanais suficientes para plotar.")
else:
    fig = px.line(
        pairwise_weekly_features.sort_values("week_start"),
        x="week_start",
        y="ndvi_mean_week",
        color="area_label",
        facet_row="comparison_pair",
        markers=True,
        title="NDVI medio semanal por area e par de comparacao",
    )
    fig.update_layout(height=700)
    fig.show()


In [ ]:
gallery_rows = sample_rows_per_area(ndvi_clean, max_images_per_area=3)
display_ndvi_gallery(gallery_rows, "Galeria NDVI por area")


In [ ]:
TARGET_DATE = "2025-12-22"
comparison_rows = closest_rows_by_date(ndvi_clean, TARGET_DATE)
display_ndvi_gallery(comparison_rows, f"Comparacao visual por data aproximada: {TARGET_DATE}", width=460)


In [ ]:
display(Markdown("## Cobertura climatica"))

weather_daily = workspace["weather_daily"]
weather_weekly = workspace["weather_weekly"]

print("Inicio do NDVI:", pd.to_datetime(ndvi_clean["date"], errors="coerce").min())
print("Inicio do clima local:", weather_daily["date"].min() if not weather_daily.empty else "sem dados")
print("Fim do clima local:", weather_daily["date"].max() if not weather_daily.empty else "sem dados")

if not weather_weekly.empty:
    fig = px.bar(
        weather_weekly,
        x="week_start",
        y="precipitation_mm_week",
        title="Precipitacao semanal da estacao local",
    )
    fig.show()


In [ ]:
display(Markdown("## Sinais operacionais por area"))

operation_columns = [
    "area_label",
    "comparison_pair",
    "harvest_yield_mean_kg_ha",
    "planting_population_mean_ha",
    "fert_dose_gap_abs_mean_kg_ha",
    "overlap_area_pct_bbox",
    "stop_duration_h_per_bbox_ha",
]

display(area_inventory[[column for column in operation_columns if column in area_inventory.columns]])


In [ ]:
display(Markdown("## MIIP e limiares"))

miip_columns = [
    "area_label",
    "comparison_pair",
    "avg_pest_count",
    "alert_hits",
    "control_hits",
    "damage_hits",
    "image_events",
]

display(area_inventory[[column for column in miip_columns if column in area_inventory.columns]])


In [ ]:
display(Markdown("## Matriz de hipoteses"))
display(hypothesis_matrix)

for row in hypothesis_matrix.itertuples(index=False):
    display(Markdown(f"### Par: `{row.pair}`"))
    display(Markdown(f"- Forca da evidencia: **{row.evidence_strength}**"))
    display(Markdown(f"- Favorece 4.0: {row.supports_4_0}"))
    display(Markdown(f"- Favorece convencional: {row.supports_convencional}"))
    display(Markdown(f"- Lacunas: {row.known_gaps}"))


In [ ]:
written_files = save_phase1_outputs(workspace, OUTPUT_DIR)
pd.DataFrame({"written_file": [str(path) for path in written_files]})
